# 50 — DeepEval Agentic Metrics
## Did the Agent Actually Succeed? — Agentic Task Evaluation
⏱ ~15 min

**Agent evaluation** is one of the hardest open problems in applied LLM engineering. Unlike RAG — where you score a single answer against retrieved context — agents make sequences of tool calls that can fail silently at any step. A single wrong tool called early in a chain can cascade into a confidently wrong final answer. By the end of this workshop you will be able to evaluate LangGraph agents systematically using DeepEval's agentic metrics.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Why Agent Eval Differs** — trajectory vs. answer evaluation |
| 2 | **Setup** — install, API key, imports |
| 3 | **The LangGraph ReAct Agent** — tools, message history |
| 4 | **Extracting Tool Calls** — parsing the message trajectory |
| 5 | **ToolCorrectnessMetric** — did it call the right tools? |
| 6 | **Deliberate Failure** — injecting misleading tasks to test the metric |
| 7 | **TaskCompletionMetric** — did the agent actually finish the job? |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` and `requirements.txt` already installed
- An `OPENAI_API_KEY` in `.env` (or Colab Secrets)
- Basic familiarity with LangGraph agents (see example `30-agentic-rag` if not)

### Key References
> Yao, S., Zhao, J., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629
>
> DeepEval documentation: https://docs.confident-ai.com/docs/metrics-tool-correctness
>
> LangGraph prebuilt ReAct agent: https://langchain-ai.github.io/langgraph/reference/prebuilt/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "deepeval",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key, (
    "OPENAI_API_KEY is not set.\n"
    "  Local: add OPENAI_API_KEY=sk-... to a .env file\n"
    "  Colab:  Secrets panel -> Add secret 'OPENAI_API_KEY'"
)
print("OPENAI_API_KEY present and loaded.")

---

## Part 1 -- Why Agent Eval Differs from RAG Eval

---

### The Core Difference

RAG evaluation scores a **single answer** against retrieved context -- one input, one output, one verdict.

Agent evaluation must score a **trajectory** -- a sequence of tool calls, observations, and reasoning steps. The final answer may look correct even when the agent took a wrong path to get there.

```
RAG:    query  ->  retrieve  ->  answer              (score the answer)

Agent:  task   ->  plan  ->  tool_1  ->  observe  ->  tool_2  ->  observe  ->  answer
                                                                     ^
                                              (score the whole trajectory)
```

---

### Why a Correct Answer Is Not Enough

Consider a task: *"Search for Python's release year and calculate 2024 minus that year."*

An agent that calls `lookup_fact("python age")` and gets `33` by coincidence produces the correct final answer -- but via the **wrong tool**. If the knowledge base is updated, the agent's shortcut breaks silently.

Evaluating the trajectory reveals these hidden failures.

---

### RAG vs Agent Evaluation -- Comparison Table

| Dimension | RAG Evaluation | Agent Evaluation |
|-----------|---------------|------------------|
| **Evaluation unit** | Single (answer) | Sequence (trajectory) |
| **Determinism** | High -- same retrieval, same answer | Low -- tool order varies |
| **Failure modes** | Irrelevant retrieval, hallucination | Wrong tool, wrong order, missed step |
| **Side effects** | None (read-only retrieval) | Yes -- tools can modify state |
| **Key metrics** | Faithfulness, Answer Relevance, Context Recall | Tool Correctness, Task Completion |
| **Ground truth** | Reference answer | Expected tool sequence |
| **Error propagation** | Isolated per query | Cascades through multi-step chains |

---

### The Agent Evaluation Flow

```
Task prompt
    |
    v
[LangGraph ReAct Agent]
    | Tool calls (search_web, calculate, lookup_fact)
    v
Message history:
  HumanMessage("Search Python's release year and calculate 2024 minus that.")
  AIMessage(tool_calls=[{"name": "search_web", "args": {...}}])
  ToolMessage("Python was first released in 1991 by Guido van Rossum.")
  AIMessage(tool_calls=[{"name": "calculate", "args": {"expression": "2024-1991"}}])
  ToolMessage("Result: 33")
  AIMessage("Python is 33 years old as of 2024.")   <- final answer (no tool_calls)
    |
    v
extract_tool_calls()  ->  tools_called = ["search_web", "calculate"]
    |
    v
[ToolCorrectnessMetric]  <-  expected_tools = ["search_web", "calculate"]
    |
    v
Score: 1.0  |  Pass: True
```

---

## Part 2 -- Setup: Tools, Agent, Helper

---

### The Three Tools

| Tool | Purpose | When to call it |
|------|---------|----------------|
| `search_web(query)` | Retrieve factual information | Factual questions about the world |
| `calculate(expression)` | Evaluate a math expression | Arithmetic or numeric transformations |
| `lookup_fact(topic)` | Query an internal knowledge base | Topic lookups in the local store |

The agent decides which tools to call and in what order -- we will verify those decisions against `expected_tools`.

In [ ]:
# ---- 2-A: Tool definitions --------------------------------------------------
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


@tool
def search_web(query: str) -> str:
    """Search the web for factual information."""
    results = {
        "python release year": "Python was first released in 1991 by Guido van Rossum.",
        "eiffel tower height": "The Eiffel Tower is 330 meters tall.",
        "largest planet": "Jupiter is the largest planet in our solar system.",
    }
    normalized_query = query.lower().replace("'", "")
    if "python" in normalized_query and "release" in normalized_query and "year" in normalized_query:
        return results["python release year"]
    for key, val in results.items():
        if key in query.lower():
            return val
    return f"Search result for: {query} -- Found relevant information."


@tool
def calculate(expression: str) -> str:
    """Evaluate a simple mathematical expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {e}"


@tool
def lookup_fact(topic: str) -> str:
    """Look up a specific fact from a knowledge base."""
    facts = {
        "python": "Python is a high-level, general-purpose programming language.",
        "langgraph": "LangGraph is a library for building stateful multi-actor LLM applications.",
        "openai": "OpenAI was founded in 2015 and created GPT-4 and ChatGPT.",
    }
    return facts.get(topic.lower(), f"No fact found for: {topic}")


TOOLS = [search_web, calculate, lookup_fact]
print(f"Tools defined: {[t.name for t in TOOLS]}")

In [ ]:
# ---- 2-B: Build the ReAct agent and define test tasks -----------------------
# LangGraph's create_react_agent wires the LLM to the tools automatically.
# The agent reasons (ReAct = Reasoning + Acting): it thinks, picks a tool,
# observes the output, thinks again, and repeats until it has a final answer.

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
AGENT_PROMPT = """You are a concise tool-using assistant. Use each needed tool at most once. After you have the required tool results, answer the user directly and do not call any more tools. For a requested arithmetic calculation, call calculate exactly once after obtaining its inputs."""
app = create_react_agent(llm, tools=TOOLS, prompt=AGENT_PROMPT)

AGENT_TASKS = [
    {
        "input": "Search for Python's release year and calculate 2024 minus that year.",
        "expected_tools": ["search_web", "calculate"],
    },
    {
        "input": "Look up a fact about LangGraph.",
        "expected_tools": ["lookup_fact"],
    },
    {
        "input": "What is the height of the Eiffel Tower? Search for it.",
        "expected_tools": ["search_web"],
    },
]

print(f"Agent ready. {len(AGENT_TASKS)} test tasks defined.")
print("Tasks:")
for i, task in enumerate(AGENT_TASKS, 1):
    print(f"  {i}. {task['input'][:70]}")
    print(f"     expected_tools: {task['expected_tools']}")

---

## Part 3 -- Extracting Tool Calls from the Message Trajectory

---

### What `app.invoke()` Returns

After running `app.invoke({"messages": [...]})`, you get a dict with a `messages` key. The message list is the full agent trajectory:

```
result["messages"] =
  [
    HumanMessage(content="Search Python's release year..."),
    AIMessage(content="", tool_calls=[{"name": "search_web", "args": {...}, "id": "..."}]),
    ToolMessage(content="Python was first released in 1991...", tool_call_id="..."),
    AIMessage(content="", tool_calls=[{"name": "calculate", "args": {...}, "id": "..."}]),
    ToolMessage(content="Result: 33", tool_call_id="..."),
    AIMessage(content="Python is 33 years old as of 2024."),  <- final answer
  ]
```

### How to Extract Tool Names

- An `AIMessage` with `.tool_calls` populated = the agent is calling a tool
- An `AIMessage` with no `.tool_calls` and non-empty `.content` = the final answer
- `ToolMessage` entries = the tool responses (not tool calls themselves)

The helper below walks the trajectory and separates these two concerns.

In [ ]:
# ---- 3-A: extract_tool_calls helper -----------------------------------------


def extract_tool_calls(messages: list) -> tuple[str, list[str]]:
    """Extract tool names called and final answer from LangGraph message history.

    Returns:
        final_answer: the last AIMessage content with no tool_calls
        tools_called: ordered list of tool names called across all AIMessages
    """
    tools_called: list[str] = []
    final_answer = ""
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tools_called.extend(tc["name"] for tc in msg.tool_calls)
        if hasattr(msg, "content") and msg.content and not getattr(msg, "tool_calls", None):
            final_answer = msg.content
    return final_answer, tools_called


print("extract_tool_calls defined.")

In [ ]:
# ---- 3-B: Run all tasks and collect trajectories ----------------------------

results = []
for task in AGENT_TASKS:
    out = app.invoke({"messages": [{"role": "user", "content": task["input"]}]})
    answer, tools = extract_tool_calls(out["messages"])
    results.append({"answer": answer, "tools_called": tools})

    match = tools == task["expected_tools"]
    status = "MATCH" if match else "MISMATCH"
    print(f"[{status}] {task['input'][:60]}")
    print(f"  Called:   {tools}")
    print(f"  Expected: {task['expected_tools']}")
    print(f"  Answer:   {answer[:80]}")
    print()

---

## Part 4 -- ToolCorrectnessMetric

---

### How It Works

`ToolCorrectnessMetric` receives:
- `tools_called` -- the list of `ToolCall` objects from the agent trajectory
- `expected_tools` -- the list of `ToolCall` objects you define

It computes a score based on how many expected tools were called, and optionally whether they were called in the right order.

### Partial Credit

`ToolCorrectnessMetric` **does give partial credit**. If a task expects `["search_web", "calculate"]` and the agent only calls `["search_web"]`, the score is approximately 0.5, not 0.0. The metric rewards getting some tools right.

This is intentional -- in multi-step tasks, partial completion is more informative than binary pass/fail.

### Key Fields in `LLMTestCase` for Agents

| Field | Type | Description |
|-------|------|-------------|
| `input` | `str` | The task prompt sent to the agent |
| `actual_output` | `str` | The agent's final text answer |
| `tools_called` | `list[ToolCall]` | Tool calls the agent made (extracted from trajectory) |
| `expected_tools` | `list[ToolCall]` | Tools you expected to be called |
| `context` | `list[str]` | Optional -- relevant context (used by other metrics) |

In [ ]:
# ---- 4-A: Build test cases and run ToolCorrectnessMetric --------------------
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import LLMTestCase, ToolCall

metric = ToolCorrectnessMetric(threshold=0.8, model="gpt-4o-mini")

test_cases = []
for task, result in zip(AGENT_TASKS, results):
    tc = LLMTestCase(
        input=task["input"],
        actual_output=result["answer"],
        tools_called=[
            ToolCall(name=t, input_parameters={}, output="")
            for t in result["tools_called"]
        ],
        expected_tools=[ToolCall(name=t) for t in task["expected_tools"]],
    )
    test_cases.append(tc)

print("=" * 65)
print(f"{'Input':<40} {'Tools':20} {'Score':6} {'Pass'}")
print("=" * 65)
for tc in test_cases:
    metric.measure(tc)
    called = [t.name for t in tc.tools_called]
    passed = metric.is_successful()
    print(f"{tc.input[:38]:<40} {str(called):<20} {metric.score:.2f}   {'YES' if passed else 'NO'}")
print("=" * 65)

---

## Part 5 -- Deliberate Failure: Misleading Task Injection

---

### Why Test Failures?

If your evaluation harness only sees tasks the agent succeeds on, you do not know if the metric is working. A good harness must include tasks designed to **fail** -- and the metric score should drop accordingly.

### Failure Injection Strategy

We phrase a calculation task as if it were a knowledge lookup. The agent may call `lookup_fact` instead of `calculate` because the phrasing resembles a factual query. This mismatch should push the `ToolCorrectnessMetric` score below 0.8.

```
Intended task:   "Calculate 2024 minus 1991"
                  -> expected: ["calculate"]

Misleading task: "What does a knowledge base say about 2024 minus 1991?"
                  -> agent may call: ["lookup_fact"]  <- WRONG
                  -> ToolCorrectnessMetric score: ~0.0  <- FAIL
```

In [ ]:
# ---- 5-A: Inject a deliberately misleading task -----------------------------
misleading_task = "What does a knowledge base say about 2024 minus 1991?"
out_fail = app.invoke({"messages": [{"role": "user", "content": misleading_task}]})
answer_fail, tools_fail = extract_tool_calls(out_fail["messages"])

print("=" * 60)
print(f"Task:            {misleading_task}")
print(f"Tools called:    {tools_fail}")
print(f"Expected:        ['calculate']")
print(f"Answer:          {answer_fail[:100]}")
print()

tc_fail = LLMTestCase(
    input=misleading_task,
    actual_output=answer_fail,
    tools_called=[ToolCall(name=t, input_parameters={}, output="") for t in tools_fail],
    expected_tools=[ToolCall(name="calculate")],
)

metric.measure(tc_fail)
print(f"ToolCorrectnessMetric score: {metric.score:.2f}")
print(f"Threshold:                   0.80")
print(f"Pass:                        {metric.is_successful()}")
print()
if not metric.is_successful():
    print("EXPECTED FAILURE -- the metric correctly detected the wrong tool.")
else:
    print("The agent happened to call the correct tool despite the misleading phrasing.")
    print("Try a more extreme mismatch to trigger the failure case.")

---

## Part 6 -- TaskCompletionMetric

---

### Two Questions, Two Metrics

| Metric | Question it answers | How it scores |
|--------|--------------------|--------------|
| `ToolCorrectnessMetric` | Did the agent call the **right tools**? | Compares tool names to `expected_tools` |
| `TaskCompletionMetric` | Did the agent **complete the task**? | LLM-judged: does the answer satisfy the task? |

These two metrics are **complementary**, not redundant:
- An agent can call all the right tools but still fail (e.g., it hallucinates in the final summary)
- An agent can complete the task via a shortcut that skips expected tools

### When to Use Each

Use **both** in a production eval harness. `ToolCorrectnessMetric` catches trajectory failures early; `TaskCompletionMetric` catches final-answer failures.

### How `TaskCompletionMetric` Works

It sends the `input` (original task) and `actual_output` (agent's final answer) to an LLM judge. The judge decides whether the task is genuinely complete -- no tool call history required. This makes it easier to compute but less precise about *where* the agent went wrong.

In [ ]:
# ---- 6-A: TaskCompletionMetric on the same test cases -----------------------
from deepeval.metrics import TaskCompletionMetric

task_metric = TaskCompletionMetric(threshold=0.7, model="gpt-4o-mini")

print("=" * 65)
print(f"{'Input':<40} {'TC Score':10} {'Pass'}")
print("=" * 65)
for tc in test_cases:
    task_metric.measure(tc)
    passed = task_metric.is_successful()
    print(f"{tc.input[:38]:<40} {task_metric.score:.2f}       {'YES' if passed else 'NO'}")
print("=" * 65)
print()
print("Comparing ToolCorrectness vs TaskCompletion for the same tasks:")
print("  Tool Correctness: did the agent follow the expected tool sequence?")
print("  Task Completion:  is the agent's final answer satisfactory to an LLM judge?")

In [ ]:
# ---- 6-B: Side-by-side comparison -- both metrics on all tasks --------------
print(f"{'Input':<38} {'TC Metric':10} {'Tool Metric':12} {'Agree?'}")
print("-" * 70)

for tc in test_cases:
    metric.measure(tc)
    task_metric.measure(tc)
    tc_pass = task_metric.is_successful()
    tool_pass = metric.is_successful()
    agree = tc_pass == tool_pass
    print(
        f"{tc.input[:36]:<38} "
        f"{task_metric.score:.2f}       "
        f"{metric.score:.2f}        "
        f"{'YES' if agree else 'DIFFER'}"
    )

print()
print("DIFFER rows are most informative: they reveal cases where")
print("task completion and tool correctness tell different stories.")

---

## Exercises

---

### Exercise 1 -- Wrong Tool on Purpose

Make the agent call `lookup_fact` on a task that should use `calculate`. Verify `ToolCorrectnessMetric` drops below the 0.8 threshold.

**Hint:** Craft a task where the phrasing strongly implies a knowledge lookup but the expected answer requires arithmetic. Use `expected_tools=["calculate"]`.

**Starter template below.**

In [ ]:
# ---- Exercise 1 Starter -----------------------------------------------------
# TODO: Replace with a task that should use 'calculate' but is phrased
#       to trick the agent into calling 'lookup_fact'.

ex1_task = "TODO: your misleading task here"
ex1_expected = ["calculate"]

# Run the agent
# ex1_out = app.invoke({"messages": [{"role": "user", "content": ex1_task}]})
# ex1_answer, ex1_tools = extract_tool_calls(ex1_out["messages"])
# print(f"Tools called: {ex1_tools}")
# print(f"Expected:     {ex1_expected}")

# Build and score the test case
# ex1_tc = LLMTestCase(
#     input=ex1_task,
#     actual_output=ex1_answer,
#     tools_called=[ToolCall(name=t, input_parameters={}, output="") for t in ex1_tools],
#     expected_tools=[ToolCall(name=t) for t in ex1_expected],
# )
# metric.measure(ex1_tc)
# print(f"Score: {metric.score:.2f}  Pass: {metric.is_successful()}")
print("Remove the comments above and fill in ex1_task to run this exercise.")

### Exercise 2 -- Add a Fourth Tool

Implement a `summarize(text: str) -> str` tool that returns the first sentence of any text passed to it. Write a task where `expected_tools = ["search_web", "summarize"]`. Verify `ToolCorrectnessMetric` passes when both tools are called.

**Starter template below.**

In [ ]:
# ---- Exercise 2 Starter -----------------------------------------------------
# TODO: Define the @tool function 'summarize'

# @tool
# def summarize(text: str) -> str:
#     """Summarize the given text by returning its first sentence."""
#     # TODO: implement
#     pass

# TODO: Rebuild the agent with the new tool
# ex2_app = create_react_agent(llm, tools=[search_web, calculate, lookup_fact, summarize])

# TODO: Write a task that requires search_web then summarize
# ex2_task = "Search for the height of the Eiffel Tower and then summarize the result."
# ex2_expected = ["search_web", "summarize"]

# TODO: Run, extract, score
print("Implement 'summarize' and rebuild the agent to run this exercise.")

### Exercise 3 -- Order Sensitivity

Set `expected_tools` in the reverse order of what the agent actually calls. Does `ToolCorrectnessMetric` give the same score, or does order matter?

**Key insight to look for:** If the score stays 1.0 regardless of order, the metric checks tool presence but not sequence. If the score drops, the metric penalizes wrong ordering.

**Starter template below.**

In [ ]:
# ---- Exercise 3 Starter -----------------------------------------------------
# Use the first task from AGENT_TASKS -- it calls [search_web, calculate] in order.
# Swap expected_tools to [calculate, search_web] and compare scores.

ex3_result = results[0]  # task 0: search_web then calculate
ex3_task = AGENT_TASKS[0]

# Correct order: [search_web, calculate]
# TODO: build tc_correct_order and measure

# Reversed order: [calculate, search_web]
# TODO: build tc_reversed_order and measure

# TODO: print both scores and conclude whether order matters
print("Build two LLMTestCase objects with tools in different orders and compare scores.")

### Exercise 4 -- End-to-End Multi-Step Pipeline

Design a task that requires the agent to call **three tools in sequence** (e.g., `lookup_fact` -> `calculate` -> `search_web`). Evaluate the full trajectory with both `ToolCorrectnessMetric` and `TaskCompletionMetric`. Do they agree?

**Starter template below.**

In [ ]:
# ---- Exercise 4 Starter -----------------------------------------------------
# TODO: Write a multi-step task. Example:
# "Look up a fact about OpenAI, then calculate how many years since 2015,
#  and finally search the web for the largest planet."

ex4_task = "TODO: your 3-step task here"
ex4_expected = ["lookup_fact", "calculate", "search_web"]  # adjust to match your task

# TODO: Run the agent
# ex4_out = app.invoke({"messages": [{"role": "user", "content": ex4_task}]})
# ex4_answer, ex4_tools = extract_tool_calls(ex4_out["messages"])

# TODO: Build the test case and score with both metrics
print("Fill in ex4_task and uncomment the code blocks to run.")

---

## Exercise Answer Key

Use this section as a reference **after** attempting the exercises yourself. These are sample solutions, not the only correct answers.

---

### Exercise 1 Answer -- Wrong Tool on Purpose

**Sample misleading task:** `"According to any knowledge source you have, what is the value of 2024 minus 1991?"`

The phrase "according to any knowledge source" strongly implies a knowledge lookup, so many agent runs will call `lookup_fact` instead of `calculate`. When the agent misses `calculate`, `ToolCorrectnessMetric` scores 0.0 -- a clear FAIL.

**What a good answer includes:**
- The agent calls `lookup_fact` (or similar) instead of `calculate`
- `ToolCorrectnessMetric` score is 0.0 or close to it
- `is_successful()` returns `False` at the 0.8 threshold

Note: The more capable the model, the harder it is to confuse. With `gpt-4o-mini`, phrasing must be quite deceptive to reliably trigger the wrong tool.

In [ ]:
# Exercise 1 -- Sample answer
ex1_task_answer = "According to any knowledge source you have, what is the value of 2024 minus 1991?"
ex1_expected_answer = ["calculate"]

ex1_out = app.invoke({"messages": [{"role": "user", "content": ex1_task_answer}]})
ex1_answer, ex1_tools = extract_tool_calls(ex1_out["messages"])

ex1_tc = LLMTestCase(
    input=ex1_task_answer,
    actual_output=ex1_answer,
    tools_called=[ToolCall(name=t, input_parameters={}, output="") for t in ex1_tools],
    expected_tools=[ToolCall(name=t) for t in ex1_expected_answer],
)
metric.measure(ex1_tc)
print(f"Tools called:  {ex1_tools}")
print(f"Expected:      {ex1_expected_answer}")
print(f"Score:         {metric.score:.2f}")
print(f"Pass (>=0.8):  {metric.is_successful()}")
print()
if not metric.is_successful():
    print("Failure injection worked -- metric correctly flagged the wrong tool.")
else:
    print("The agent still called 'calculate' despite the phrasing -- try a stronger mismatch.")

### Exercise 2 Answer -- Add a Fourth Tool

**Key points:**
- Rebuild the agent with `tools=[search_web, calculate, lookup_fact, summarize]`
- The task must explicitly instruct the agent to summarize (otherwise it skips the tool)
- `ToolCorrectnessMetric` should score 1.0 when both `search_web` and `summarize` are called

**Partial credit note:** If only `search_web` is called (agent skips summarize), score is approximately 0.5.

In [ ]:
# Exercise 2 -- Sample answer


@tool
def summarize(text: str) -> str:
    """Summarize the given text by returning its first sentence."""
    sentences = text.split(". ")
    return sentences[0] + "." if sentences else text


ex2_app = create_react_agent(llm, tools=[search_web, calculate, lookup_fact, summarize])

ex2_task = (
    "Search for the height of the Eiffel Tower, "
    "then call the summarize tool on the search result."
)
ex2_expected = ["search_web", "summarize"]

ex2_out = ex2_app.invoke({"messages": [{"role": "user", "content": ex2_task}]})
ex2_answer, ex2_tools = extract_tool_calls(ex2_out["messages"])

ex2_tc = LLMTestCase(
    input=ex2_task,
    actual_output=ex2_answer,
    tools_called=[ToolCall(name=t, input_parameters={}, output="") for t in ex2_tools],
    expected_tools=[ToolCall(name=t) for t in ex2_expected],
)
metric.measure(ex2_tc)
print(f"Tools called:  {ex2_tools}")
print(f"Expected:      {ex2_expected}")
print(f"Score:         {metric.score:.2f}")
print(f"Pass (>=0.8):  {metric.is_successful()}")

### Exercise 3 Answer -- Order Sensitivity

**Finding:** `ToolCorrectnessMetric` (as of DeepEval 1.x) checks tool **presence**, not strict sequence order. Reversing `expected_tools` typically produces the **same score** as the correct order.

**Implication:** If your task has strict ordering requirements (e.g., you must authenticate before querying), `ToolCorrectnessMetric` alone is insufficient. Combine it with a custom `GEvalMetric` that checks ordering in its evaluation criteria.

**What good output looks like:**
- Correct order score approximately equals reversed order score (both high)
- This confirms the metric is order-agnostic

In [ ]:
# Exercise 3 -- Sample answer
ex3_result = results[0]  # task 0: search_web then calculate
ex3_base_task = AGENT_TASKS[0]

# Correct order
tc_correct_order = LLMTestCase(
    input=ex3_base_task["input"],
    actual_output=ex3_result["answer"],
    tools_called=[
        ToolCall(name=t, input_parameters={}, output="")
        for t in ex3_result["tools_called"]
    ],
    expected_tools=[ToolCall(name="search_web"), ToolCall(name="calculate")],
)

# Reversed order
tc_reversed_order = LLMTestCase(
    input=ex3_base_task["input"],
    actual_output=ex3_result["answer"],
    tools_called=[
        ToolCall(name=t, input_parameters={}, output="")
        for t in ex3_result["tools_called"]
    ],
    expected_tools=[ToolCall(name="calculate"), ToolCall(name="search_web")],
)

metric.measure(tc_correct_order)
score_correct = metric.score

metric.measure(tc_reversed_order)
score_reversed = metric.score

print(f"Score (correct order  [search_web, calculate]):  {score_correct:.2f}")
print(f"Score (reversed order [calculate, search_web]):  {score_reversed:.2f}")
print()
if abs(score_correct - score_reversed) < 0.05:
    print("Conclusion: ToolCorrectnessMetric is ORDER-AGNOSTIC.")
    print("Use a custom GEvalMetric if strict ordering must be enforced.")
else:
    print("Conclusion: ToolCorrectnessMetric penalized order reversal.")

### Exercise 4 Answer -- End-to-End Multi-Step Pipeline

In [ ]:
# Exercise 4 -- Sample answer
ex4_task_answer = (
    "Look up a fact about OpenAI. "
    "Then calculate how many years have passed since 2015. "
    "Finally, search the web for the largest planet in the solar system."
)
ex4_expected_answer = ["lookup_fact", "calculate", "search_web"]

ex4_out = app.invoke({"messages": [{"role": "user", "content": ex4_task_answer}]})
ex4_answer, ex4_tools = extract_tool_calls(ex4_out["messages"])

ex4_tc = LLMTestCase(
    input=ex4_task_answer,
    actual_output=ex4_answer,
    tools_called=[ToolCall(name=t, input_parameters={}, output="") for t in ex4_tools],
    expected_tools=[ToolCall(name=t) for t in ex4_expected_answer],
)

metric.measure(ex4_tc)
task_metric.measure(ex4_tc)

print(f"Tools called:             {ex4_tools}")
print(f"Expected:                 {ex4_expected_answer}")
print(f"ToolCorrectnessMetric:    {metric.score:.2f}  Pass: {metric.is_successful()}")
print(f"TaskCompletionMetric:     {task_metric.score:.2f}  Pass: {task_metric.is_successful()}")
print()
agree = metric.is_successful() == task_metric.is_successful()
print(f"Both metrics agree: {agree}")
if not agree:
    print("Disagreement detected -- check whether the agent completed each sub-task.")

---

## What's Next?

You now have a systematic harness for evaluating LangGraph agents with DeepEval. Here is where to take it further:

### Deepen the Evaluation
- **Example 49 -- G-Eval Custom Criteria** (`../49-deepeval-geval-custom/`): Build a custom G-Eval metric that checks agent reasoning quality, not just tool names. Use it to penalize agents that call the right tools for the wrong reasons.
- **Extend `ToolCorrectnessMetric`**: Add `input_parameters` and `output` to `ToolCall` objects so the metric can verify argument values as well as tool names.

### Scale the Harness
- **Parameterize tasks from a YAML file**: Define `(input, expected_tools)` pairs externally so domain experts can add eval cases without touching Python.
- **Run with `deepeval test run`**: Export test cases to DeepEval's CI integration for continuous evaluation on every deploy.

### Go Multi-Turn
- **Example 51 -- Conversational Eval** (`../51-deepeval-conversational/`): Extend this harness to multi-turn conversations where the expected tool sequence spans several user turns.

### Combine with RAG
- **Example 30 -- Agentic RAG** (`../30-agentic-rag/`): The agent in that example uses a retrieval tool -- apply `ToolCorrectnessMetric` there to verify the agent calls the retriever before the LLM.

---

### Further Reading
- Yao et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* https://arxiv.org/abs/2210.03629
- DeepEval agentic metrics: https://docs.confident-ai.com/docs/metrics-tool-correctness
- LangGraph documentation: https://langchain-ai.github.io/langgraph/

---

*Workshop complete. The recommended next step is example 49 -- custom G-Eval criteria for domain-specific agent quality.*